# Error Handling in Express.js

> Error-handling middleware is defined by having **exactly four arguments**: `(err, req, res, next)`. Express inspects `fn.length` to detect this. Three arguments = normal middleware. Five = ignored. If you drop `next` because your linter says it's unused, **the handler silently stops being an error handler.**

---

## 1. Structure of the error middleware

Error middleware must be registered **last**, after every route and every other `app.use()`. Express walks the stack in registration order, so anything declared before your routes can't catch their errors.

```js
const express = require('express');
const app = express();

// ... routes and regular middleware go here ...

// 1. Fallback for 404 (Not Found)
app.use((req, res, next) => {
  const error = new Error(`Not Found - ${req.originalUrl}`);
  error.statusCode = 404;
  next(error);
});

// 2. Global error handler — must have 4 parameters
app.use((err, req, res, next) => {
  // If the response already started streaming, hand off to Express's
  // built-in handler, which will close the connection.
  if (res.headersSent) {
    return next(err);
  }

  const statusCode = err.statusCode || 500;

  res.status(statusCode).json({
    status: 'error',
    message: err.message || 'Internal Server Error',
    ...(process.env.NODE_ENV !== 'production' && { stack: err.stack }),
  });
});
```

> [!tip] Prefer `err.statusCode` over `res.statusCode`
> A common pattern is `res.status(404)` in the 404 handler, then reading `res.statusCode === 200 ? 500 : res.statusCode` in the global handler. It works, but it couples the status to response state that any earlier middleware could have mutated. Attaching the code to the **error object** is self-contained and survives being re-thrown across modules.

### What Express does if you don't write one

Express ships a default error handler. It sets the status from `err.status`/`err.statusCode` (or 500), sets `err.headersSent` handling, and — in non-production — dumps the **full stack trace into the HTTP response body**. That's the leak you're preventing by writing your own.

---

## 2. Synchronous vs. asynchronous errors

### Synchronous code

Express wraps sync handlers in a try/catch internally. Throwing just works.

```js
app.get('/sync-error', (req, res) => {
  throw new Error('This sync error is caught automatically!');
});
```

### Asynchronous code — Express 4

Express 4 predates promises in its routing layer. A rejected promise inside an `async` handler is **not** caught, becomes an unhandled rejection, and in modern Node versions **crashes the process**. Catch manually and forward via `next()`.

```js
app.get('/async-error', async (req, res, next) => {
  try {
    const data = await database.findUser();
    res.json(data);
  } catch (error) {
    next(error);
  }
});
```

### Asynchronous code — Express 5

Express 5 (now the default `express` install on npm) awaits handler return values natively. Rejected promises and `async` throws propagate down the middleware chain on their own — no try/catch, no wrapper.

```js
// Express 5 — this is enough
app.get('/async-error', async (req, res) => {
  const data = await database.findUser();
  res.json(data);
});
```

> [!warning] Callbacks are still not covered
> Even in Express 5, an error thrown inside a **non-promise callback** (`fs.readFile(path, (err, data) => { throw ... })`) escapes the request context entirely. Promisify it (`fs/promises`) or call `next(err)` from inside the callback.

---

## 3. Streamline async handlers (Express 4)

Writing try/catch around every DB call is boilerplate bloat. Wrap it once:

```js
const asyncHandler = (fn) => (req, res, next) => {
  Promise.resolve(fn(req, res, next)).catch(next);
};

app.get('/users', asyncHandler(async (req, res) => {
  const users = await database.getUsers();
  res.json(users);
}));
```

Alternatives:

| Approach | Notes |
|---|---|
| Hand-rolled `asyncHandler` | Zero deps, four lines, easy to reason about. |
| `express-async-errors` | Monkey-patches Express 4's Layer so no wrapper is needed. `require` it once at the top of your entry file. |
| `express-async-handler` | Same idea as the hand-rolled version, published as a package. |
| Upgrade to Express 5 | Removes the problem instead of patching it. |

---

## 4. Operational custom error class

A plain `Error` carries no HTTP semantics. Subclassing lets you pass structured context across modules.

```js
class AppError extends Error {
  constructor(message, statusCode) {
    super(message);
    this.statusCode = statusCode;
    this.status = `${statusCode}`.startsWith('4') ? 'fail' : 'error';
    this.isOperational = true; // predictable user error, not a system bug

    Error.captureStackTrace(this, this.constructor);
  }
}

app.get('/premium-content', (req, res, next) => {
  const isSubscribed = false;
  if (!isSubscribed) {
    return next(new AppError('Subscription required to access this resource', 403));
  }
  res.send('Welcome to the VIP zone!');
});
```

### Why `isOperational` matters

It splits errors into two categories that deserve different treatment:

- **Operational** — expected failures in a correct program: bad user input, 404, failed auth, third-party API timeout. Send a clean message to the client. Keep running.
- **Programmer** — bugs: `undefined` is not a function, bad `await`, null deref. The process is now in an unknown state. Log loudly, send a generic 500, and consider restarting.

```js
if (err.isOperational) {
  res.status(err.statusCode).json({ status: err.status, message: err.message });
} else {
  logger.error(err); // full detail to logs only
  res.status(500).json({ status: 'error', message: 'Something went wrong' });
}
```

---

## 5. Normalizing library errors (Mongoose / MongoDB)

Mongoose throws its own error shapes. Left alone, they surface as ugly 500s with internal field paths exposed. Convert them to `AppError` *before* the generic branch runs.

```js
const handleCastError = (err) =>
  new AppError(`Invalid ${err.path}: ${err.value}`, 400);

const handleDuplicateKey = (err) => {
  const field = Object.keys(err.keyValue)[0];
  return new AppError(`Duplicate value for '${field}'. Please use another.`, 400);
};

const handleValidationError = (err) => {
  const messages = Object.values(err.errors).map((e) => e.message);
  return new AppError(`Invalid input: ${messages.join('. ')}`, 400);
};

app.use((err, req, res, next) => {
  let error = err;

  if (err.name === 'CastError')       error = handleCastError(err);
  if (err.code === 11000)             error = handleDuplicateKey(err);
  if (err.name === 'ValidationError') error = handleValidationError(err);
  if (err.name === 'JsonWebTokenError')  error = new AppError('Invalid token', 401);
  if (err.name === 'TokenExpiredError') error = new AppError('Token expired', 401);

  // ... then the send logic from section 4
});
```

| Mongoose/Mongo error | Trigger | Map to |
|---|---|---|
| `CastError` | `/users/abc` where `abc` isn't a valid ObjectId | 400 |
| `ValidationError` | Schema validator failed on save | 400 |
| `MongoServerError` code `11000` | Unique index violation | 400 or 409 |
| `DocumentNotFoundError` | `.orFail()` on a query with no match | 404 |

> [!note] `findById` returns `null`, it doesn't throw
> A valid-but-nonexistent ID gives you `null`. That's a 404 you have to raise yourself: `if (!doc) return next(new AppError('Not found', 404));` — or use `.orFail()`.

---

## 6. Process-level safety nets

Express middleware only sees errors that happen *during a request*. Everything else needs a catch at the process level, in your entry file.

```js
// Sync throws that escaped everything. Process state is unreliable — exit.
process.on('uncaughtException', (err) => {
  logger.error('UNCAUGHT EXCEPTION', err);
  process.exit(1);
});

const server = app.listen(PORT);

// Rejected promise with no .catch() — e.g. DB connection failure at boot
process.on('unhandledRejection', (err) => {
  logger.error('UNHANDLED REJECTION', err);
  server.close(() => process.exit(1)); // drain in-flight requests, then die
});

// Container/orchestrator shutdown signal
process.on('SIGTERM', () => {
  server.close(() => {
    logger.info('Process terminated');
  });
});
```

> [!important] Exiting is the correct move
> After an `uncaughtException` the process may be holding half-mutated state, leaked handles, or a broken DB connection. Don't try to "recover." Let a supervisor (nodemon in dev, PM2 / Docker / systemd in prod) restart it clean.

Register `uncaughtException` **before** everything else so it's live during startup.

---

## 7. Logging

`console.log` is not a logging strategy — it has no levels, no structure, and no transport.

```js
const pino = require('pino');
const logger = pino({ level: process.env.LOG_LEVEL || 'info' });
```

In the error handler:

- Log the **full error object** (message, stack, and any custom fields) server-side.
- Send the client only what it needs to act on.
- Attach a request ID so a user-reported "error id: 4f2a..." maps to a log line.

```js
const { randomUUID } = require('crypto');

app.use((req, res, next) => {
  req.id = randomUUID();
  next();
});

// in the error handler
logger.error({ err, reqId: req.id, url: req.originalUrl, method: req.method });
res.status(500).json({ status: 'error', message: 'Something went wrong', reqId: req.id });
```

---

## 8. Validate input at the edge

Most 500s are really 400s that got in. Validate before the handler body runs, so bad input becomes a clean operational error.

```js
const Joi = require('joi');

const validate = (schema) => (req, res, next) => {
  const { error, value } = schema.validate(req.body, { abortEarly: false });
  if (error) {
    return next(new AppError(error.details.map((d) => d.message).join(', '), 400));
  }
  req.body = value;
  next();
};

app.post('/users', validate(userSchema), asyncHandler(createUser));
```

(`zod` and `express-validator` are the other common picks — same pattern.)

---

## 9. Scoped error handlers

You can register more than one error handler. Mount a router-level one to give a subsystem its own error shape:

```js
apiRouter.use((err, req, res, next) => {
  if (!err.statusCode) return next(err);   // not mine — bubble up
  res.status(err.statusCode).json({ error: err.message });
});

app.use('/api', apiRouter);
app.use(globalErrorHandler); // still last
```

Calling `next(err)` from inside an error handler skips all remaining *normal* middleware and jumps to the next *error* handler.

---

## 10. Gotchas checklist

- [ ] Handler has all **four** params, and `next` is not stripped by lint autofix.
- [ ] Error middleware is registered **after** every route.
- [ ] `res.headersSent` is checked before writing.
- [ ] Stack traces are gated on `NODE_ENV !== 'production'`.
- [ ] Every `async` route in Express 4 is wrapped (or you're on Express 5).
- [ ] `return next(err)` — the `return` prevents execution continuing past the branch.
- [ ] `next(err)` is called with a real `Error`, not a string. `next('foo')` is treated as a **route name**, not an error, and behaves nothing like you expect.
- [ ] `next()` with **no arguments** means "continue to the next middleware" — passing anything at all (except the literal `'route'`) diverts to the error chain.
- [ ] Mongoose `CastError` / `11000` are mapped, not leaked.
- [ ] `unhandledRejection` and `uncaughtException` are registered.
- [ ] Errors are logged server-side with full detail, not just returned.

---

## 11. Minimal complete wiring

```js
// app.js
const express = require('express');
const AppError = require('./utils/AppError');
const globalErrorHandler = require('./controllers/errorController');

const app = express();
app.use(express.json());

app.use('/api/v1/users', require('./routes/userRoutes'));

// 404 — anything unmatched falls through to here
app.all('*', (req, res, next) => {
  next(new AppError(`Can't find ${req.originalUrl} on this server`, 404));
});

app.use(globalErrorHandler);

module.exports = app;
```

> [!note] Express 5 wildcard syntax
> Express 5 replaced the path-to-regexp version underneath. `app.all('*')` throws on some 5.x setups; use `app.use()` with no path, or the named wildcard `app.all('/*splat', ...)`.

---

## 12. Testing error paths

```js
const request = require('supertest');

it('returns 404 for unknown route', async () => {
  const res = await request(app).get('/nope');
  expect(res.status).toBe(404);
  expect(res.body.message).toMatch(/Can't find/);
});

it('does not leak stack traces in production', async () => {
  process.env.NODE_ENV = 'production';
  const res = await request(app).get('/route-that-throws');
  expect(res.body.stack).toBeUndefined();
});
```

Error paths are the least-exercised code in most codebases and the most likely to be broken when you finally need them. Test at minimum: 404, validation failure, duplicate key, and one deliberate 500.

---

## Related

- [[Express Middleware]]
- [[Mongoose Schema Validation]]
- [[Node.js Async Patterns]]
- [[HTTP Status Codes]]